#STURM-fusion

In [1]:
#@title Imports

import random
import os
import sys
import zipfile
import ee
import time

from dataclasses import dataclass
from pathlib import Path
from google.colab import auth, drive

In [2]:
#@title Setup

root_path = "/content/drive/MyDrive/MSc/STURM-fusion"  #@param {type:"string", multiline:true}
mount_point = "/content/drive"  #@param {type:"string"}
clone_repo = False  #@param {type:"boolean"}
reset_export = False  #@param {type:"boolean"}
gee_export = False  #@param {type:"boolean"}
push_to_HF = False  #@param {type:"boolean"}
cancle_gee_tasks = False  #@param {type:"boolean"}
gee_project = "243624085884"  #@param {type:"string"}

# Google Drive must be mounted: GEE batch exports can only write to Drive,
# so the pipeline needs a synced view of it to read the exported files back.
drive.mount(mount_point)

repo_url = "https://github.com/TAX2310/STURM-fusion.git"

project_root = os.path.join(root_path, "STURM-fusion") if clone_repo else root_path

if clone_repo and not os.path.exists(project_root):
    !git clone {repo_url} "{project_root}"

project_root = Path(project_root)
assert project_root.exists(), f"Repo not found at {project_root}. Enable clone_repo or fix root_path."

if gee_export or cancle_gee_tasks:
    ee.Authenticate(force=True, auth_mode="notebook")
    ee.Initialize(project=gee_project)

sys.path.append(str(project_root))

from src.config import CFG
cfg = CFG()
cfg.ROOT = project_root
cfg.DRIVE_ROOT = Path(mount_point) / "MyDrive"
cfg.GEE_PROJECT = gee_project

print("ROOT:", cfg.ROOT)
print("DRIVE_ROOT:", cfg.DRIVE_ROOT)
print("GEE project:", cfg.GEE_PROJECT)

In [3]:
requirements = cfg.ROOT / "requirements.txt"
!pip install -r {requirements}

from src.gee.tasks import cancel_all_tasks
from src.util.io import clear_export_folder, create_dataset_structure, save_dataframe_to_csv, zip_dataset
from src.data.sturm_flood import download_and_extract
from src.gee.tasks import has_active_tasks
from src.pipeline.matching import process_csv
from src.pipeline.export import export_all_s1_images
from src.pipeline.assemble import assemble_dataset
from src.pipeline.preprocessing import preprocessing_s1_pipeline, preprocessing_s2_pipeline
from src.pipeline.validation import validate_files, validate_nan_files, remove_bad_nan_files, validate_dataset
from src.util.metrics import check_image_shapes, get_band_min_max, get_band_percentiles, get_max_time_difference_with_row
from src.hugging_face.push_dataset import push_zip_to_hf

In [4]:
if cancle_gee_tasks:
  cancel_all_tasks()

In [5]:
if reset_export:
    clear_export_folder(cfg)

create_dataset_structure(cfg)

In [6]:
#@title Download STURM-Flood

download_and_extract(cfg)

In [7]:
if gee_export and not has_active_tasks():
    print("No active GEE tasks, starting")
    images, df_fusion = process_csv(cfg.OLD_S2_METADATA_CSV, cfg, verbose=False)
    print('')
    print(len(images), "images processed.")

    save_dataframe_to_csv(df_fusion, cfg.NEW_METADATA_CSV)
    print('New Metadata saved')

    export_all_s1_images(images, cfg)
    print('S1 images sent for exported')
else:
    print("Active GEE tasks detected. Please wait for them to finish before startingt.")

In [8]:
while not validate_files(cfg):
    assemble_dataset(cfg)

preprocessing_s1_pipeline(cfg)
print('S1 preprocessing done')

preprocessing_s2_pipeline(cfg)
print('S2 preprocessing done')

In [9]:

if not validate_nan_files(cfg):
    df = remove_bad_nan_files(cfg)

In [10]:
print('Starting validation Pipeline')
if validate_dataset(cfg):

    print('Starting inspection Pipeline')

    result = check_image_shapes(cfg.NEW_S1_PATH, cfg.NEW_S2_PATH)
    print(result)

    get_band_min_max(cfg.NEW_S1_PATH)
    get_band_percentiles(cfg.NEW_S1_PATH)

    max_s2_diff = get_max_time_difference_with_row(cfg.NEW_METADATA_CSV, sentinel_timestamp="sentinel2_timestamp")

    print(f"Max S2 time difference: {max_s2_diff['time_diff_hours']:.2f} hours")

    max_s1_diff = get_max_time_difference_with_row(cfg.NEW_METADATA_CSV, sentinel_timestamp="sentinel1_timestamp")

    print(f"Max S1 time difference: {max_s1_diff['time_diff_hours']:.2f} hours")

In [11]:
if push_to_HF and validate_dataset(cfg):
    zip_dataset(cfg)
    print('Dataset zipped')
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")

    from huggingface_hub import login

    login(token=HF_TOKEN)

    url = push_zip_to_hf(zip_path=cfg.NEW_ZIP_PATH,repo_id=cfg.HF_REPO_ID,path_in_repo="Dataset.zip",private=False,)
    print(url)